# prime-rl S3: reverse-text RL smoke (RTX 6000 Pro offline, 3-process)

The unified `rl` entrypoint requires `>=2 GPUs` (CHANGELOG 2026-02-23 removed the
single-GPU overlap mode). On Kaggle's single RTX 6000 Pro we therefore run the
three RL components as **separate processes that share GPU 0**:

1. **inference** (vLLM server) — background, `gpu_memory_utilization=0.5`
2. **orchestrator** — background, no GPU (talks to vLLM over HTTP)
3. **trainer** — foreground, uses the remaining ~50% of VRAM

Inference and trainer coexist because vLLM is capped to half the VRAM.
Orchestrator and trainer share state via filesystem (default broadcast type).

Tiny config: `max_steps=2 batch_size=4 rollouts=2 seq_len=256`.
Success: trainer reports 2 RL steps and exits 0.

## 1. Install prime-rl (same recipe as S1/S2)

In [ ]:
import os, subprocess, glob, re

PREP = "/kaggle/input/notebooks/<KAGGLE_USER>/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP
WHEELS = f"{BASE}/wheels"
MODEL_DIR = f"{BASE}/models/Qwen3-0.6B"

SKIP_PREFIXES = ("torch-", "torchvision-", "torchaudio-", "nvidia_")

def parse_version(name):
    m = re.match(r"([a-zA-Z0-9_]+?)-(\d+(?:\.\d+)*)", os.path.basename(name))
    if not m:
        return None, None
    return m.group(1).lower(), tuple(int(x) for x in m.group(2).split("."))

all_wheels = sorted(glob.glob(f"{WHEELS}/*.whl"))
remaining = [w for w in all_wheels if not os.path.basename(w).startswith(SKIP_PREFIXES)]
keep = {}
for w in remaining:
    pkg, ver = parse_version(w)
    if pkg is None:
        keep[os.path.basename(w)] = (None, w); continue
    if pkg in keep and keep[pkg][0] is not None:
        if ver > keep[pkg][0]:
            keep[pkg] = (ver, w)
    else:
        keep[pkg] = (ver, w)
filtered = [v[1] for v in keep.values()]
print(f"installing {len(filtered)} wheels")

subprocess.run(
    "python3.12 -m pip install --no-index --no-build-isolation --pre --no-deps "
    + " ".join(filtered),
    shell=True, check=True,
)

# Patch the installed reverse_text.py to load the dataset from a local directory
# instead of `PrimeIntellect/Reverse-Text-RL` on the Hub. The local snapshot is
# bundled by the prep notebook at output/datasets/Reverse-Text-RL/.
RT_DATA = f"{BASE}/datasets/Reverse-Text-RL"
RT_PY = "/usr/local/lib/python3.12/dist-packages/reverse_text.py"
if os.path.exists(RT_PY) and os.path.isdir(RT_DATA):
    subprocess.run(
        f"sed -i 's|PrimeIntellect/Reverse-Text-RL|{RT_DATA}|g' {RT_PY}",
        shell=True, check=True,
    )
    print(f"patched {RT_PY} -> {RT_DATA}")
    subprocess.run(f"grep -n 'load_dataset' {RT_PY} | head -3", shell=True)
else:
    print(f"WARN: cannot patch reverse_text.py (RT_PY exists={os.path.exists(RT_PY)}, RT_DATA exists={os.path.isdir(RT_DATA)})")

## 2. Sanity: GPU + reverse-text Environment importable?

In [ ]:
import subprocess
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)
print()
# Look for the reverse-text env package
subprocess.run("python3.12 -m pip list 2>/dev/null | grep -iE '(reverse|verifiers|prime-)' | head", shell=True)
print()
try:
    import importlib
    m = importlib.import_module("reverse_text")
    print(f"reverse_text module loaded from {m.__file__}")
except Exception as e:
    print(f"reverse_text import error: {e}")
    # Some env packages register via entry points rather than top-level modules

## 3. Write minimal RL TOML

Based on `examples/reverse_text/rl.toml` but with smaller batch/rollouts and the
local model path.

In [ ]:
import os, subprocess

PREP = "/kaggle/input/notebooks/<KAGGLE_USER>/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP
MODEL_DIR = f"{BASE}/models/Qwen3-0.6B"

OUT_DIR = "/kaggle/working/run_default"
os.makedirs(OUT_DIR, exist_ok=True)

TRAINER_TOML = f'''max_steps = 2

[model]
name = "{MODEL_DIR}"
seq_len = 256

[optim]
lr = 3e-6

[ckpt]
'''

INFER_TOML = f'''gpu_memory_utilization = 0.5

[model]
name = "{MODEL_DIR}"
max_model_len = 256
enforce_eager = true

[server]
port = 8000
'''

# Disable filters (the default zero_advantage filter rejects every rollout when
# the base Qwen3-0.6B can't reverse text -- all rewards collapse to 0, so the
# group has no advantage variance and orchestrator gives up at step 0).
# max_completion_tokens=128 because the env wraps output in <reversed_text>...
# tags that don't fit in 32 tokens.
ORCH_TOML = f'''max_steps = 2
batch_size = 4
seq_len = 256
rollouts_per_example = 2
filters = []

[model]
name = "{MODEL_DIR}"

[train.sampling]
max_completion_tokens = 128

[[train.env]]
id = "reverse-text"
'''

os.makedirs("/kaggle/working", exist_ok=True)
with open("/kaggle/working/trainer.toml", "w") as f:
    f.write(TRAINER_TOML)
with open("/kaggle/working/infer.toml", "w") as f:
    f.write(INFER_TOML)
with open("/kaggle/working/orch.toml", "w") as f:
    f.write(ORCH_TOML)

for p in ["/kaggle/working/trainer.toml", "/kaggle/working/infer.toml", "/kaggle/working/orch.toml"]:
    print(f"=== {p} ===")
    subprocess.run(f"cat {p}", shell=True)
    print()

## 4. Run RL (2 steps, single GPU)

PR #971 single-GPU flags: trainer and inference share GPU 0, vLLM gets 50% of
VRAM (well under our 102GB). max_model_len kept tiny for the smoke.

In [ ]:
import os, subprocess, time, urllib.request, urllib.error, signal

LOG_DIR = "/kaggle/working/proc_logs"
os.makedirs(LOG_DIR, exist_ok=True)

API_KEY = "dummy"

PREP = "/kaggle/input/notebooks/<KAGGLE_USER>/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP

# HF_DATASETS_CACHE must be writable -- the datasets library creates a lock file
# during builder init even when the source data is on a read-only mount. Point
# it at /kaggle/working (writable). HF_HOME points at the prep cache for any
# Hub-side lookups.
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)

env = os.environ.copy()
env.update({
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "WANDB_MODE": "disabled",
    "CUDA_VISIBLE_DEVICES": "0",
    "VLLM_API_KEY": API_KEY,
    "HF_HOME": f"{BASE}/hf_cache",
    "HF_HUB_CACHE": f"{BASE}/hf_cache",
    "HF_DATASETS_CACHE": "/kaggle/working/hf_cache",
})
print(f"HF_HOME (read)  = {BASE}/hf_cache")
print(f"HF_DATASETS_CACHE (write) = /kaggle/working/hf_cache")

procs = {}

def start_bg(name, cmd):
    log_path = f"{LOG_DIR}/{name}.log"
    log_f = open(log_path, "wb")
    p = subprocess.Popen(cmd, shell=True, env=env, stdout=log_f, stderr=subprocess.STDOUT)
    procs[name] = (p, log_f, log_path)
    print(f"[{name}] started pid={p.pid}, log={log_path}")
    return p

def cleanup():
    for name, (p, f, _) in procs.items():
        if p.poll() is None:
            print(f"[{name}] terminating pid={p.pid}")
            try:
                p.terminate(); p.wait(timeout=10)
            except Exception:
                p.kill()
        f.close()

def tail(path, n=40):
    try:
        return subprocess.check_output(f"tail -n {n} {path}", shell=True).decode("utf-8", errors="replace")
    except Exception as e:
        return f"<tail error: {e}>"

try:
    start_bg(
        "inference",
        "python3.12 -m prime_rl.entrypoints.inference @ /kaggle/working/infer.toml",
    )

    print("\nWaiting for vLLM to come up at http://localhost:8000 ...")
    ready = False
    deadline = time.time() + 600
    while time.time() < deadline:
        if procs["inference"][0].poll() is not None:
            raise RuntimeError(
                f"inference died early. Tail of log:\n{tail(procs['inference'][2], 80)}"
            )
        try:
            req = urllib.request.Request(
                "http://localhost:8000/v1/models",
                headers={"Authorization": f"Bearer {API_KEY}"},
            )
            with urllib.request.urlopen(req, timeout=5) as r:
                if r.status == 200:
                    ready = True
                    break
        except Exception:
            pass
        time.sleep(5)
    if not ready:
        raise RuntimeError(
            f"inference did not become ready in 10 min. Tail:\n{tail(procs['inference'][2], 80)}"
        )
    print("inference is up.")

    start_bg(
        "orchestrator",
        "python3.12 -m prime_rl.orchestrator.orchestrator @ /kaggle/working/orch.toml",
    )
    time.sleep(5)
    if procs["orchestrator"][0].poll() is not None:
        raise RuntimeError(
            f"orchestrator died early. Tail:\n{tail(procs['orchestrator'][2], 80)}"
        )
    print("orchestrator is up.")

    print("\nStarting trainer (foreground via torchrun, output live via tee)...")
    print("=" * 60)
    trainer_log = f"{LOG_DIR}/trainer.log"
    t0 = time.time()
    trainer_cmd = (
        "set -o pipefail; "
        "python3.12 -m torch.distributed.run --nproc-per-node 1 "
        "-m prime_rl.trainer.rl.train @ /kaggle/working/trainer.toml "
        f"2>&1 | tee {trainer_log}"
    )
    trainer = subprocess.run(["bash", "-c", trainer_cmd], env=env)
    elapsed = time.time() - t0
    print("=" * 60)
    print(f"\ntrainer exit={trainer.returncode}  elapsed={elapsed:.1f}s")

    if trainer.returncode != 0:
        print("\n--- inference log tail ---")
        print(tail(procs["inference"][2], 40))
        print("\n--- orchestrator log tail ---")
        print(tail(procs["orchestrator"][2], 40))
        raise RuntimeError(f"trainer failed with exit code {trainer.returncode}")

    print("\nS3 PASS")
finally:
    cleanup()